In [1]:
import pandas as pd
import json
import shutil
import os


In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Pick the dataset with the DATASET env var: make process-data DATASET=olma
DATASET = os.environ.get('DATASET', 'epfl')
dataset_dir = os.path.join('./datasets', DATASET)
if not os.path.isdir(dataset_dir):
    raise ValueError(f"Unknown dataset '{DATASET}', no folder {dataset_dir}")

def find_file(name):
    # A dataset file wins over the shared one
    for folder in [dataset_dir, './shared']:
        path = os.path.join(folder, name)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{name} not found in {dataset_dir} or ./shared")

beverage_list_path = find_file('TCF-Coffe_App-Export_Beverage-List.csv')
impact_data_path = find_file('TCF-Coffe_App-Export_Impact-Data.csv')
sugar_data_path = find_file('TCF-Coffe_App-Export_Sugar-Data.csv')
impact_description_path = find_file('TCF-Coffe_App-Export_Impact-Description.csv')
coffee_description_path = find_file('TCF-Coffe_App-Export_Coffee-Description.csv')

df_beverage_list = pd.read_csv(beverage_list_path,delimiter=';',encoding='windows-1252')
df_impact_data = pd.read_csv(impact_data_path,delimiter=';',encoding='windows-1252')
df_sugar_data = pd.read_csv(sugar_data_path,delimiter=';',encoding='windows-1252')
df_impact_description = pd.read_csv(impact_description_path,delimiter=';',encoding='windows-1252')
df_coffee_description = pd.read_csv(coffee_description_path,delimiter=';',encoding='windows-1252')

# The sale point is only useful in the name when there is more than one
if df_beverage_list['Salepoint'].nunique() > 1:
    df_beverage_list['Retail name'] = df_beverage_list['Retail name'] + " " + df_beverage_list['Salepoint']

df_beverage_list

,Beverage ID,Retail name,Retail price,Hidden Costs,True Price,Net Hidden Costs,Beverage-type,Salepoint,Deca,Milk Type,Price Excluding Tax,Value Added Tax,Smart Value Added Tax,Smart Pricing rounded,Offsetting,Labels
0,"Café Conventionnel, Brazil","Café Conventionnel, Brazil",4.5,0.209,4.71,0.209,Café,OLMA,False,none,4.163,0.337,0.052,4.40,0,none
1,"Café Conventionnel, Colombia","Café Conventionnel, Colombia",4.5,0.262,4.76,0.262,Café,OLMA,False,none,4.163,0.337,0.058,4.40,0,none
2,"Café Conventionnel, India","Café Conventionnel, India",4.5,0.396,4.90,0.396,Café,OLMA,False,none,4.163,0.337,0.075,4.45,0,none
3,"Café Conventionnel, Indonesia","Café Conventionnel, Indonesia",4.5,0.355,4.86,0.355,Café,OLMA,False,none,4.163,0.337,0.070,4.45,0,none
4,"Café Conventionnel, Vietnam","Café Conventionnel, Vietnam",4.5,0.176,4.68,0.176,Café,OLMA,False,none,4.163,0.337,0.048,4.35,0,none
5,"Café Fairtrade, Brazil","Café Fairtrade, Brazil",4.5,0.183,4.68,0.183,Café,OLMA,False,none,4.163,0.337,0.049,4.35,0,fairtrade
6,"Café Fairtrade, Colombia","Café Fairtrade, Colombia",4.5,0.228,4.73,0.228,Café,OLMA,False,none,4.163,0.337,0.054,4.40,0,fairtrade
7,"Café Fairtrade, India","Café Fairtrade, India",4.5,0.361,4.86,0.361,Café,OLMA,False,none,4.163,0.337,0.071,4.45,0,fairtrade
8,"Café Fairtrade, Indonesia","Café Fairtrade, Indonesia",4.5,0.269,4.77,0.269,Café,OLMA,False,none,4.163,0.337,0.059,4.40,0,fairtrade
9,"Café Fairtrade, Vietnam","Café Fairtrade, Vietnam",4.5,0.160,4.66,0.160,Café,OLMA,False,none,4.163,0.337,0.046,4.35,0,fairtrade


In [4]:
os.makedirs('./results', exist_ok=True)

duplicate_rows = df_impact_data.groupby(by=["Beverage ID", "Ingredient", "Impact Category", "Stage", "Indicator"]).filter(lambda x: len(x) >= 2)
duplicate_rows = duplicate_rows.sort_values(by=["Beverage ID", "Ingredient", "Impact Category", "Stage", "Indicator"])
duplicate_rows.to_csv(f'./results/duplicate_rows_{DATASET}.csv',index=False)

In [5]:
duplicate_rows

,Beverage ID,Ingredient,Stage,Impact Category,Indicator,Unit,Monetary Value,Value,References
878,Cappuccino,Cow milk,Transport,Environment,Fine particulate matter formation,kg PM10 eq,0.000000,0.000036,"Road; transport, freight, light commercial veh..."
879,Cappuccino,Cow milk,Transport,Environment,Fine particulate matter formation,kg PM2.5 eq,0.001708,0.000018,"Road; transport, freight, lorry with refrigera..."


In [6]:
def get_indicator_values(row):

    # # Find matches where df_impact_description['Indicator'] is contained in row['Indicator']
    # matched_rows = df_impact_description[
    #     df_impact_description['Indicator'].apply(lambda x: x in row['Indicator'])
    # ]
    
    # if matched_rows.empty:
    #     impact_definition = None
    #     monetisation_method = None
    # else:
    #     # Select the most specific match (longest string in df_impact_description['Indicator'])
    #     impact_definition = matched_rows.loc[matched_rows['Indicator'].str.len().idxmax(), 'Indicator definition']
    #     monetisation_method = matched_rows.loc[matched_rows['Indicator'].str.len().idxmax(), 'Monetisation method']
    
    return {
        'indicators': row['Indicator'],
        'unit': row['Unit'],
        'impactValue': row['Value'], 
        'costValue': row['Monetary Value'], 
        # 'impactDefinition': impact_definition,
        # 'monetisationMethod': monetisation_method,
        'reference': None if pd.isna(row['References']) else row['References']
    }
    
def process_group(keys, x):
    ingredient, stage, category = keys
    return {
        'stage': stage,
        'ingredient': ingredient,
        'impactCategory': category,
        'impactValue': x['Value'].sum(),   # Sum impact values
        'costValue': x['Monetary Value'].sum(),   # Sum cost values
        'details': x.apply(get_indicator_values, axis=1).tolist()
    }

def calculate_impacts(group):
    # Group by ingredient, stage and impact category.
    # The three names come from the group key, not from the rows: since pandas
    # 3.0 the columns we group on are not passed to the group anymore.
    grouped = group.groupby(['Ingredient', 'Stage', 'Impact Category'])
    return [process_group(keys, x) for keys, x in grouped]


In [7]:
# Delete everything inside ./results/<dataset>
results_dir = os.path.join('./results', DATASET)
if os.path.exists(results_dir):
    shutil.rmtree(results_dir)
os.makedirs(results_dir)

In [8]:
# Transform the data
import re


formatted_data = []

i = 0;

# milk_beverage_lists = ['Cappuccino',"Chocolait","Latte Macchiato","Macchiato","Mocaccino", 'Renversé']

        
for _, beverage in df_beverage_list.iterrows():
    beverage_id = beverage['Beverage ID']
    retail_name = beverage['Retail name']

    # all_removals = labels_list + ['décaféiné', "lait d'amande", "lait d'avoine", "lait de soja", "sans-lactose"]

    # recipe_id_pattern = r'(' + '|'.join(map(re.escape, all_removals)) + r')'
    # recipe_id = re.sub(recipe_id_pattern, '', retail_name, flags=re.IGNORECASE).replace(',','').strip()
    recipe_id = beverage['Beverage-type']
    # sale_point_id = sale_point_mapping[beverage['Salepoint']]
    # is_decaf = 'décaféiné' in beverage['Retail name'].lower()
    is_decaf = beverage['Deca']
    # has_milk = any(x in beverage['Retail name'] for x in milk_beverage_lists)

    # Determine the type of milk, if present
    # milk_types = {'lait d\'amande': 'Almond', 'lait d\'avoine': 'Oat', 'lait de soja': 'Soy', 'sans-lactose': 'Lactose-Free'}
    # milk_type = None
    # for key, value in milk_types.items():
    #     if key in beverage['Retail name'].lower():
    #         milk_type = value
    #         break
    # milk_type = milk_type if milk_type else 'Dairy' if any(x in beverage['Retail name'].lower() for x in ['milk', 'latte', 'cappuccino', 'renversé']) else None
        
    milk_type = beverage['Milk Type']
    has_milk = milk_type != 'none'

    definition_row = df_coffee_description[df_coffee_description['Recipe'] == recipe_id]
    definition = definition_row['Definition'].iloc[0] if not definition_row.empty else ""


    impact_rows = df_impact_data[df_impact_data['Beverage ID'] == beverage_id]
    # print(impact_rows)
    ingredient_list = impact_rows['Ingredient'].unique().tolist()

    if impact_rows.empty:
        print(f"No impact data found for {beverage_id}")

    grouped_impacts = calculate_impacts(impact_rows)

    path_impacts = results_dir + '/impacts/'+beverage_id.lower().replace(' ','_').replace(',','')+'.json'
    print(path_impacts)
    os.makedirs(os.path.dirname(path_impacts), exist_ok=True)
    with open(path_impacts, 'w', encoding='utf-8') as f:
        json.dump(grouped_impacts, f, indent=4, ensure_ascii=False)



    formatted_data.append({
        'serveId': beverage_id,
        'recipeId': recipe_id,
        'retailName': retail_name,
        'retailPrice': beverage['Retail price'],
        'hiddenCost': beverage['Hidden Costs'],
        'truePrice': beverage['True Price'],
        'labels': beverage['Labels'],
        "priceWithoutTax": beverage["Price Excluding Tax"],
        'valueAddedTax': beverage["Value Added Tax"],
        'smartValueAddedTax': beverage["Smart Value Added Tax"],
        'smartPricingRounded': beverage["Smart Pricing rounded"],
        "offsetting": beverage["Offsetting"],
        'isDecaf': is_decaf,
        'hasMilk': has_milk,
        'milkType': milk_type,
        'coffeeDetails': definition, 

    })

# Create the formatted DataFrame
formatted_df = pd.DataFrame(formatted_data)

# Save to CSV
output_path = results_dir + '/coffee_data.csv'
formatted_df.to_csv(output_path, index=False)



print(f"Formatted data saved to {output_path}")

./results/olma/impacts/café_conventionnel_brazil.json
./results/olma/impacts/café_conventionnel_colombia.json
./results/olma/impacts/café_conventionnel_india.json
./results/olma/impacts/café_conventionnel_indonesia.json
./results/olma/impacts/café_conventionnel_vietnam.json
./results/olma/impacts/café_fairtrade_brazil.json
./results/olma/impacts/café_fairtrade_colombia.json
./results/olma/impacts/café_fairtrade_india.json
./results/olma/impacts/café_fairtrade_indonesia.json
./results/olma/impacts/café_fairtrade_vietnam.json
./results/olma/impacts/cappuccino.json
./results/olma/impacts/cappuccino_lait_d'amande.json
./results/olma/impacts/cappuccino_lait_d'avoine.json
./results/olma/impacts/cappuccino_lait_de_soja.json
Formatted data saved to ./results/olma/coffee_data.csv


In [9]:

list_sugars = ["Swiss sugar default","Swiss sugar low","Swiss sugar moderate","Swiss sugar high"]

for sugar in list_sugars:
    sugar_rows = df_sugar_data[df_sugar_data['Beverage ID'] == sugar]
    print(sugar_rows)

    grouped_impacts = calculate_impacts(sugar_rows)

    path_impacts = results_dir + '/sugar/'+sugar.lower().replace(' ','_').replace(',','')+'.json'
    os.makedirs(os.path.dirname(path_impacts), exist_ok=True)
    with open(path_impacts, 'w', encoding='utf-8') as f:
        json.dump(grouped_impacts, f, indent=4, ensure_ascii=False)


             Beverage ID      Country Ingredient Labels        Stage  \
0    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
1    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
2    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
3    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
4    Swiss sugar default  Switzerland  Sugarbeet   none   Production   
..                   ...          ...        ...    ...          ...   
182  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
183  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
184  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
185  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   
186  Swiss sugar default  Switzerland  Sugarbeet   none  Consumption   

    Impact Category                          Indicator         Unit  Value  \
0       Environment  Fine particulate matter formation  k

In [10]:
output_path= results_dir + '/impacts_definitions.csv'
df_impact_description.to_csv(output_path, index=False)

In [11]:
source_dir = results_dir
destination_dir = os.path.join('../public/data/', DATASET)

# Clean only this dataset in the destination, other datasets stay
if os.path.exists(destination_dir):
    shutil.rmtree(destination_dir)

shutil.copytree(source_dir, destination_dir)

print(f"Contents of {source_dir} copied to {destination_dir}")

Contents of ./results/olma copied to ../public/data/olma
